In [1]:
!ls

ckpt/       main.ipynb:Zone.Identifier  preprocess/  train.py:Zone.Identifier
eval.py     pixi.lock                   src/
main.ipynb  pixi.toml                   train.py


In [2]:
import os
import torch

from src.preprocess import convert_to_mindrecord
from src.dataset import create_dataset
from src.seq2seq import Seq2Seq, WithLossCell, InferCell
from src.config import cfg

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


### train

In [4]:
os.makedirs(cfg.dataset_path, exist_ok=True)
os.makedirs(cfg.ckpt_save_path, exist_ok=True)
convert_to_mindrecord('src/cmn_zhsim.txt', cfg.dataset_path, cfg.max_seq_length)

en_vocab_size: 1153
ch_vocab_size: 1116


(1153, 1116)

In [5]:
ds_train = create_dataset(cfg.dataset_path, cfg.batch_size)

In [6]:
network = Seq2Seq(cfg)
network = WithLossCell(network, cfg).to(device)
optimizer = torch.optim.Adam(network.parameters(), lr=cfg.learning_rate, betas=(0.9, 0.98))

In [7]:
for epoch in range(1, cfg.num_epochs + 1):
    network.train()
    for step, data in enumerate(ds_train, start=1):
        src = data['encoder_data'].to(device)
        dst = data['decoder_data'].to(device)
        label = data['target_data'].to(device)

        optimizer.zero_grad()
        loss = network(src, dst, label)
        loss.backward()
        optimizer.step()

        if step % cfg.save_checkpoint_steps == 0:
            ckpt_path = os.path.join(cfg.ckpt_save_path, f'gru-{epoch}_{step}.ckpt')
            torch.save({'model_state_dict': network._backbone.state_dict()}, ckpt_path)
            print(f'epoch: {epoch} step: {step}, loss is {loss.item():.7f}')

epoch: 1 step: 125, loss is 1.8444529
epoch: 2 step: 125, loss is 1.5364015
epoch: 3 step: 125, loss is 0.8060730
epoch: 4 step: 125, loss is 0.6828192
epoch: 5 step: 125, loss is 0.4447956
epoch: 6 step: 125, loss is 0.2391970
epoch: 7 step: 125, loss is 0.2072638
epoch: 8 step: 125, loss is 0.1239359
epoch: 9 step: 125, loss is 0.1716198
epoch: 10 step: 125, loss is 0.1069309
epoch: 11 step: 125, loss is 0.0765421
epoch: 12 step: 125, loss is 0.1413512
epoch: 13 step: 125, loss is 0.0847386
epoch: 14 step: 125, loss is 0.0480313
epoch: 15 step: 125, loss is 0.0631750


### eval

In [8]:
ds_eval = create_dataset(cfg.dataset_path, cfg.eval_batch_size, is_training=False)
infer_net = Seq2Seq(cfg, is_train=False)
infer_net = InferCell(infer_net, cfg).to(device)
checkpoint = torch.load(cfg.checkpoint_path, map_location=device)
infer_net.network.load_state_dict(checkpoint.get('model_state_dict', checkpoint))
infer_net.eval()

InferCell(
  (network): Seq2Seq(
    (encoder): Encoder(
      (embedding): Embedding(1154, 1024)
      (gru): GRU(
        (rnn): GRU(1024, 1024)
      )
    )
    (decoder): Decoder(
      (embedding): Embedding(1116, 1024)
      (gru): GRU(
        (rnn): GRU(1024, 1024)
      )
      (dense): Linear(in_features=1024, out_features=1116, bias=True)
      (softmax): LogSoftmax(dim=2)
    )
  )
)

In [9]:
with open(os.path.join(cfg.dataset_path, 'en_vocab.txt'), 'r', encoding='utf-8') as f:
    en_vocab = f.read().split('\n')

with open(os.path.join(cfg.dataset_path, 'ch_vocab.txt'), 'r', encoding='utf-8') as f:
    ch_vocab = f.read().split('\n')

In [10]:
with torch.no_grad():
    for data in ds_eval:
        en_data = ''
        ch_data = ''

        for x in data['encoder_data'][0].tolist():
            if x == 0:
                break
            en_data += en_vocab[x] + ' '

        for x in data['decoder_data'][0].tolist():
            if x == 0:
                break
            if x == 1:
                continue
            ch_data += ch_vocab[x]

        output = infer_net(data['encoder_data'].to(device), data['decoder_data'].to(device))
        out = ''
        for x in output[0].tolist():
            if x == 0:
                break
            out += ch_vocab[x]

        print('English:', en_data)
        print('expect Chinese:', ch_data)
        print('predict Chinese:', out)
        print(' ')

English: go away . 
expect Chinese: 走开！
predict Chinese: 走开！
 
English: i like chocolate . 
expect Chinese: 我喜欢巧克力。
predict Chinese: 我喜欢巧克力。
 
English: i want a guitar . 
expect Chinese: 我想要一把吉他。
predict Chinese: 我想要一把吉他。
 
English: well let s go . 
expect Chinese: 好吧，我们走吧。
predict Chinese: 好吧，我们走吧。
 
English: you are lying . 
expect Chinese: 你在撒谎。
predict Chinese: 你在撒谎。
 
English: we d better talk . 
expect Chinese: 我们谈谈比较好。
predict Chinese: 我们谈谈比较好。
 
English: i was surprised . 
expect Chinese: 我吃惊了。
predict Chinese: 我吃惊了。
 
English: this book will do . 
expect Chinese: 这本书就行了。
predict Chinese: 这本书就行了。
 
English: i m still puzzled . 
expect Chinese: 我还不明白。
predict Chinese: 我还不明白。
 
English: keep the dog out . 
expect Chinese: 别让狗进来。
predict Chinese: 别让狗进来。
 
English: you can tell us . 
expect Chinese: 你能告诉我们。
predict Chinese: 你能告诉我们。
 
English: let s take a rest . 
expect Chinese: 我们休息一下吧。
predict Chinese: 我们休息一下吧。
 
English: where do we go ? 
expect Chinese: 我们去哪儿？
predict Chinese: 